In [1]:
import pandas as pd
import numpy as np
import os
import random
import warnings
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_transformer.csv'

# Transformer用ハイパーパラメータ
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 前処理 (PID Features採用)
# ==============================================================================
def add_features(df):
    df_eng = df.copy()
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    grouped = df_eng.groupby('sample_id')
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    for col in base_cols:
        df_eng[f'{col}_w'] = df_eng[col] * gain
        df_eng[f'{col}_d1'] = grouped[col].diff().fillna(0) * gain
        # Transformerは位置情報(Positional Encoding)を使いますが
        # 累積和(積分)を入れておくとトレンドを掴みやすくなります
        df_eng[f'{col}_cum'] = grouped[col].cumsum()
        
    return df_eng

print(">>> Loading & Preprocessing...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

train = add_features(train)
test = add_features(test)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        x = group[self.feature_cols].values
        seq_len = len(x)
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        ret = {'x': torch.tensor(x_pad), 'mask': torch.tensor(mask), 'id': sample_id}
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
        return ret

# ==============================================================================
# 4. Model Definition: Transformer (Encoder Only)
# ==============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=40, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        
        # Embedding (特徴量を d_model 次元へ射影)
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )
        
        # Positional Encoding (時間の概念を付与)
        self.pos_encoder = PositionalEncoding(d_model, max_len=MAX_LEN, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model*4, 
            dropout=dropout,
            batch_first=True,
            norm_first=True # Pre-Norm (学習安定化のため)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output Head
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: [Batch, Time, Features]
        
        # マスク作成 (Padding部分をAttentionさせない)
        # xからpadding maskを作るのが定石ですが、
        # 今回は簡易的に全結合させ、Loss側でマスク処理します。
        
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        
        out = self.head(x)
        return out.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold CV with Transformer...")

test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, 'lever', MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, 'lever', MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = TransformerModel(
        input_dim=len(feature_cols),
        d_model=128,  # 次元数
        nhead=4,      # ヘッド数
        num_layers=3  # 層数
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_transformer_fold{fold}.pth"
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # Inference
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for t, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
        
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_transformer_fold{fold+1}.csv', index=False)
        
    except Exception as e:
        print(f"  Error in Inference Fold {fold+1}: {e}")

# ==============================================================================
# 6. Final Submission
# ==============================================================================
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions generated.")

Using Device: cuda
>>> Loading & Preprocessing...

>>> Starting 5-Fold CV with Transformer...

=== Fold 1/5 ===
  Ep  5 | Train: 0.9025 | Val: 1.0640
  Ep 10 | Train: 0.7724 | Val: 1.0232
  Ep 15 | Train: 0.6355 | Val: 1.0151
  Ep 20 | Train: 0.4918 | Val: 1.0569
  Ep 25 | Train: 0.4222 | Val: 1.0933
  Ep 30 | Train: 0.3992 | Val: 1.1045
  >> Best Val: 1.01514

=== Fold 2/5 ===
  Ep  5 | Train: 0.9476 | Val: 0.9174
  Ep 10 | Train: 0.8014 | Val: 0.8773
  Ep 15 | Train: 0.6449 | Val: 0.9303
  Ep 20 | Train: 0.5040 | Val: 0.9383
  Ep 25 | Train: 0.4248 | Val: 0.9646
  Ep 30 | Train: 0.4058 | Val: 0.9762
  >> Best Val: 0.87731

=== Fold 3/5 ===
  Ep  5 | Train: 0.9477 | Val: 0.9540
  Ep 10 | Train: 0.7817 | Val: 0.9944
  Ep 15 | Train: 0.6430 | Val: 0.9929
  Ep 25 | Train: 0.4315 | Val: 1.0050
  Ep 30 | Train: 0.4070 | Val: 1.0207
  >> Best Val: 0.92727

=== Fold 4/5 ===
  Ep  5 | Train: 0.9281 | Val: 1.0170
  Ep 10 | Train: 0.7938 | Val: 0.9363
  Ep 15 | Train: 0.6585 | Val: 0.9972
  Ep 